In [1]:
# ResNet experiment: label smoothing 0.05
# No pretrained weights. Only permitted competition training data.
#
# References:
# ResNet: https://arxiv.org/abs/1512.03385
# Cutout: https://arxiv.org/abs/1708.04552
# EMA implementation reference:
# https://pytorch.org/docs/stable/optim.html#weight-averaging-swa-and-ema

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import copy
import hashlib
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import display, FileLink


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42
SPLIT_SEED = 42           # Keep 42 for comparison with your baseline.
EPOCHS = 200
BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128

LEARNING_RATE = 0.1
WEIGHT_DECAY = 5e-4
MOMENTUM = 0.9
PCT_START = 0.15

VALIDATION_SIZE = 5000
WIDTHS = (42, 84, 168, 336)
BLOCKS = (2, 2, 2, 2)

LABEL_SMOOTHING = 0.05    # Controlled change: baseline used 0.10.
CUTOUT = 16
EMA_DECAY = 0.999

EXPERIMENT = "four_stage_cutout_ema_ls005"
DATA_OVERRIDE = None     # Optionally set the competition directory.

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable a GPU in Kaggle notebook settings, "
    "then restart and run this cell."
)

device = torch.device("cuda:0")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUT = Path("/kaggle/working") / f"{EXPERIMENT}_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("PyTorch:", torch.__version__, flush=True)
print("Output:", OUT, flush=True)
print("One GPU is used.", flush=True)
print("Label smoothing:", LABEL_SMOOTHING, flush=True)


# ============================================================
# 2. Find and validate competition metadata
# ============================================================

if DATA_OVERRIDE is not None:
    DATA = Path(DATA_OVERRIDE)
else:
    preferred = Path(
        "/kaggle/input/competitions/"
        "cifar-10-res-net-architecture-challenge-dl"
    )
    if (preferred / "train_labels.csv").is_file():
        DATA = preferred
    else:
        candidates = [
            p.parent
            for p in Path("/kaggle/input").rglob("train_labels.csv")
            if (p.parent / "test.csv").is_file()
            and (p.parent / "sample_submission.csv").is_file()
            and (p.parent / "train_images").is_dir()
            and (p.parent / "test_images").is_dir()
        ]
        assert len(candidates) == 1, (
            f"Found {len(candidates)} possible datasets. "
            "Set DATA_OVERRIDE explicitly."
        )
        DATA = candidates[0]

train_df = pd.read_csv(DATA / "train_labels.csv")
test_df = pd.read_csv(DATA / "test.csv")
sample_df = pd.read_csv(DATA / "sample_submission.csv")

assert {"id", "filename", "label"}.issubset(train_df.columns)
assert {"id", "filename"}.issubset(test_df.columns)
assert sample_df.columns.tolist() == ["id", "label"]

assert len(train_df) == 50000
assert train_df[["id", "filename", "label"]].notna().all().all()
assert test_df[["id", "filename"]].notna().all().all()
assert sample_df["id"].notna().all()

assert train_df["id"].is_unique
assert train_df["filename"].is_unique
assert test_df["id"].is_unique
assert test_df["filename"].is_unique
assert sample_df["id"].is_unique

assert len(test_df) == len(sample_df)
assert set(test_df["id"]) == set(sample_df["id"])

numeric_labels = pd.to_numeric(train_df["label"], errors="raise")
assert np.equal(numeric_labels, np.floor(numeric_labels)).all()
labels = numeric_labels.to_numpy(dtype=np.int64)
assert ((labels >= 0) & (labels <= 9)).all()

print("Data:", DATA, flush=True)
print("Training images:", len(train_df), flush=True)
print("Evaluation images:", len(test_df), flush=True)


# ============================================================
# 3. Load training images and create the fixed split
# ============================================================

def load_images(frame, folder):
    base = DATA / folder

    def read_one(filename):
        with Image.open(base / str(filename)) as image:
            array = np.asarray(image.convert("RGB"), dtype=np.uint8)
        if array.shape != (32, 32, 3):
            raise ValueError(
                f"Unexpected image shape: {filename}: {array.shape}"
            )
        return array

    with ThreadPoolExecutor(max_workers=8) as pool:
        return np.stack(list(pool.map(read_one, frame["filename"])))


print("Loading permitted training images...", flush=True)
train_images = load_images(train_df, "train_images")

training_idx, validation_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=VALIDATION_SIZE,
    stratify=labels,
    random_state=SPLIT_SEED,
)

assert len(np.intersect1d(training_idx, validation_idx)) == 0

np.savez(
    OUT / "data_split.npz",
    training=training_idx,
    validation=validation_idx,
)

# Estimate normalization from the fitting split only.
# Float64 channel sums avoid making a large float64 image copy.
channel_sum = np.zeros(3, dtype=np.float64)
channel_square_sum = np.zeros(3, dtype=np.float64)
pixel_count = 0

for start in range(0, len(training_idx), 1000):
    chunk = train_images[
        training_idx[start:start + 1000]
    ].astype(np.float64) / 255.0
    channel_sum += chunk.sum(axis=(0, 1, 2))
    channel_square_sum += (chunk * chunk).sum(axis=(0, 1, 2))
    pixel_count += chunk.shape[0] * 32 * 32

mean_np = channel_sum / pixel_count
std_np = np.sqrt(
    np.maximum(channel_square_sum / pixel_count - mean_np**2, 1e-12)
)

mean = torch.tensor(
    mean_np, dtype=torch.float32, device=device
).view(1, 3, 1, 1)

std = torch.tensor(
    std_np, dtype=torch.float32, device=device
).view(1, 3, 1, 1)

# Keep images as uint8 on GPU; convert only individual batches.
all_images = torch.from_numpy(
    train_images.transpose(0, 3, 1, 2).copy()
).to(device)

all_labels = torch.from_numpy(labels).to(device)
train_indices_gpu = torch.as_tensor(training_idx, device=device)
val_indices_gpu = torch.as_tensor(validation_idx, device=device)

del train_images

print("Fitting images:", len(training_idx), flush=True)
print("Validation images:", len(validation_idx), flush=True)
print("Mean:", mean_np, flush=True)
print("Std:", std_np, flush=True)


# ============================================================
# 4. Training-only augmentation
# ============================================================

coordinates = torch.arange(32, device=device)


def prepare_batch(images, augment=False):
    x = images.float().div_(255.0)
    x = (x - mean) / std

    if not augment:
        return x.contiguous(memory_format=torch.channels_last)

    batch = len(x)

    # Horizontal flip.
    flip = torch.rand(batch, device=device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)

    # Independent random 32x32 crops after reflect padding.
    padded = F.pad(x, (4, 4, 4, 4), mode="reflect")
    row_start = torch.randint(0, 9, (batch,), device=device)
    col_start = torch.randint(0, 9, (batch,), device=device)

    rows = row_start[:, None] + coordinates
    cols = col_start[:, None] + coordinates

    x = padded[
        torch.arange(batch, device=device)[:, None, None, None],
        torch.arange(3, device=device)[None, :, None, None],
        rows[:, None, :, None],
        cols[:, None, None, :],
    ]

    # Cutout: exactly CUTOUT pixels wide before boundary clipping.
    if CUTOUT > 0:
        cy = torch.randint(0, 32, (batch, 1, 1), device=device)
        cx = torch.randint(0, 32, (batch, 1, 1), device=device)
        yy = coordinates.view(1, 32, 1)
        xx = coordinates.view(1, 1, 32)

        top = cy - CUTOUT // 2
        left = cx - CUTOUT // 2

        erased = (
            (yy >= top)
            & (yy < top + CUTOUT)
            & (xx >= left)
            & (xx < left + CUTOUT)
        )
        x = x.masked_fill(erased[:, None], 0.0)

    return x.contiguous(memory_format=torch.channels_last)


# ============================================================
# 5. Four-stage residual CNN
# ============================================================

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(
            cin, cout, 3, stride=stride, padding=1, bias=False
        )
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(
            cout, cout, 3, padding=1, bias=False
        )
        self.b2 = nn.BatchNorm2d(cout)

        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride=stride, bias=False),
                nn.BatchNorm2d(cout),
            )

    def forward(self, x):
        residual = x if self.sc is None else self.sc(x)
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return F.relu(out + residual)


class FourStageResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, WIDTHS[0], 3, padding=1, bias=False),
            nn.BatchNorm2d(WIDTHS[0]),
            nn.ReLU(),
        )

        layers = []
        cin = WIDTHS[0]

        for stage, (cout, count) in enumerate(zip(WIDTHS, BLOCKS)):
            for block in range(count):
                stride = 2 if stage > 0 and block == 0 else 1
                layers.append(BasicBlock(cin, cout, stride))
                cin = cout

        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(cin, 10)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)


model = FourStageResNet().to(
    device=device, memory_format=torch.channels_last
)

parameter_count = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
assert parameter_count == 4_815_940, parameter_count
assert parameter_count <= 5_000_000

print(f"Trainable parameters: {parameter_count:,}", flush=True)
print("Parameter budget check: PASSED", flush=True)


# ============================================================
# 6. EMA, optimizer and scheduler
# ============================================================

ema = copy.deepcopy(model).eval()
ema.requires_grad_(False)
ema_updates = 0


@torch.no_grad()
def update_ema():
    global ema_updates
    ema_updates += 1

    decay = min(
        EMA_DECAY,
        (1.0 + ema_updates) / (10.0 + ema_updates),
    )

    source = model.state_dict()
    for name, target in ema.state_dict().items():
        value = source[name]
        if target.is_floating_point():
            target.mul_(decay).add_(value, alpha=1.0 - decay)
        else:
            target.copy_(value)


decay_parameters = []
no_decay_parameters = []

for parameter in model.parameters():
    if parameter.ndim > 1:
        decay_parameters.append(parameter)
    else:
        no_decay_parameters.append(parameter)

optimizer = torch.optim.SGD(
    [
        {"params": decay_parameters, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    nesterov=True,
)

steps_per_epoch = math.ceil(len(training_idx) / BATCH_SIZE)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=EPOCHS * steps_per_epoch,
    pct_start=PCT_START,
)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.amp.GradScaler("cuda")


# ============================================================
# 7. Metadata and safe checkpoint saving
# ============================================================

fingerprint = hashlib.sha256(
    (DATA / "train_labels.csv").read_bytes()
).hexdigest()

config = {
    "experiment": EXPERIMENT,
    "architecture": "four_stage_basic_resnet",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "validation_size": VALIDATION_SIZE,
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "widths": list(WIDTHS),
    "blocks": list(BLOCKS),
    "label_smoothing": LABEL_SMOOTHING,
    "cutout": CUTOUT,
    "color_jitter": 0.0,
    "use_ema": True,
    "ema_decay": EMA_DECAY,
    "ema_buffers": "average_floating_copy_integer",
    "ema_warmup": "min(decay, (1+n)/(10+n))",
    "horizontal_flip_averaging": True,
    "optimizer": "SGD",
    "momentum": MOMENTUM,
    "nesterov": True,
    "scheduler": "OneCycleLR",
    "pct_start": PCT_START,
    "pretrained": False,
    "parameters": parameter_count,
    "mean": mean_np.tolist(),
    "std": std_np.tolist(),
    "train_labels_csv_sha256": fingerprint,
    "torch_version": str(torch.__version__),
    "checkpoint_selection": "highest EMA flip-TTA validation accuracy",
    "cudnn_benchmark": True,
}

(OUT / "config.json").write_text(json.dumps(config, indent=2))


def atomic_save(value, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)


def cpu_state(network):
    return {
        name: value.detach().cpu().clone()
        for name, value in network.state_dict().items()
    }


# ============================================================
# 8. Validation / inference
# ============================================================

@torch.inference_mode()
def predict(network, images, indices=None):
    network.eval()
    outputs = []

    count = len(images) if indices is None else len(indices)

    for start in range(0, count, EVAL_BATCH_SIZE):
        if indices is None:
            batch = images[start:start + EVAL_BATCH_SIZE]
        else:
            idx = indices[start:start + EVAL_BATCH_SIZE]
            batch = images[idx]

        x = prepare_batch(batch, augment=False)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = network(x)
            flipped_logits = network(
                x.flip(3).contiguous(memory_format=torch.channels_last)
            )

        probabilities = (
            logits.float().softmax(dim=1)
            + flipped_logits.float().softmax(dim=1)
        ) * 0.5

        outputs.append(probabilities.cpu().numpy())

    return np.concatenate(outputs, axis=0)


# ============================================================
# 9. Training
# ============================================================

history = []
best_accuracy = -1.0
best_epoch = 0

print("Training from fresh random initialization.", flush=True)

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    model.train()

    order = train_indices_gpu[
        torch.randperm(len(train_indices_gpu), device=device)
    ]

    loss_sum = 0.0
    seen = 0
    skipped_steps = 0
    starting_lr = optimizer.param_groups[0]["lr"]

    for start in range(0, len(order), BATCH_SIZE):
        indices = order[start:start + BATCH_SIZE]

        x = prepare_batch(all_images[indices], augment=True)
        y = all_labels[indices]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
            loss = criterion(logits, y)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}. "
                f"Last checkpoint: {OUT / 'last.pt'}"
            )

        scaler.scale(loss).backward()

        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()

        # If AMP skips an optimizer update, skip EMA/scheduler too.
        if scaler.get_scale() >= previous_scale:
            update_ema()
            scheduler.step()
        else:
            skipped_steps += 1

        loss_sum += loss.detach().item() * len(y)
        seen += len(y)

    val_probabilities = predict(
        ema, all_images, indices=val_indices_gpu
    )
    val_predictions = val_probabilities.argmax(axis=1)
    validation_accuracy = float(
        (val_predictions == labels[validation_idx]).mean()
    )

    if validation_accuracy > best_accuracy:
        best_accuracy = validation_accuracy
        best_epoch = epoch

        atomic_save(
            {
                "model": cpu_state(ema),
                "config": config,
                "epoch": epoch,
                "validation_accuracy": best_accuracy,
                "weight_source": "ema",
            },
            OUT / "best.pt",
        )

        np.save(OUT / "val_probs.npy", val_probabilities)
        np.save(
            OUT / "val_ids.npy",
            train_df.iloc[validation_idx]["id"].to_numpy(),
        )

        pd.DataFrame({
            "id": train_df.iloc[validation_idx]["id"].to_numpy(),
            "true_label": labels[validation_idx],
            "prediction": val_predictions,
        }).to_csv(OUT / "validation_predictions.csv", index=False)

    row = {
        "epoch": epoch,
        "learning_rate": starting_lr,
        "loss": loss_sum / seen,
        "validation_accuracy": validation_accuracy,
        "best_validation_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "skipped_amp_steps": skipped_steps,
        "seconds": time.time() - started,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(
        OUT / "training_log.csv", index=False
    )

    # Recovery checkpoint, saved after every epoch.
    atomic_save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "ema": ema.state_dict(),
            "ema_updates": ema_updates,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_validation_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "config": config,
            "history": history,
            "training_indices": training_idx,
            "validation_indices": validation_idx,
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
        },
        OUT / "last.pt",
    )

    remaining_hours = (
        np.mean([item["seconds"] for item in history[-10:]])
        * (EPOCHS - epoch)
        / 3600
    )

    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"LR {starting_lr:.6f} | "
        f"Loss {row['loss']:.4f} | "
        f"Val {validation_accuracy:.2%} | "
        f"Best {best_accuracy:.2%} | "
        f"{row['seconds']:.0f}s | "
        f"Remaining ~{remaining_hours:.2f}h",
        flush=True,
    )


# ============================================================
# 10. Reload and verify this run's selected checkpoint
# ============================================================

# Load only your own trusted checkpoint.
checkpoint = torch.load(
    OUT / "best.pt",
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(checkpoint["model"])
model.eval()

# The final inference model is a single ResNet.
# EMA is no longer needed.
del ema

val_probabilities = predict(
    model, all_images, indices=val_indices_gpu
)
val_predictions = val_probabilities.argmax(axis=1)
verified_accuracy = float(
    (val_predictions == labels[validation_idx]).mean()
)

assert abs(
    verified_accuracy - checkpoint["validation_accuracy"]
) < 1e-12

np.save(OUT / "val_probs.npy", val_probabilities)
np.save(
    OUT / "val_ids.npy",
    train_df.iloc[validation_idx]["id"].to_numpy(),
)

pd.DataFrame({
    "id": train_df.iloc[validation_idx]["id"].to_numpy(),
    "true_label": labels[validation_idx],
    "prediction": val_predictions,
}).to_csv(OUT / "validation_predictions.csv", index=False)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

class_rows = []
for class_id, class_name in enumerate(class_names):
    mask = labels[validation_idx] == class_id
    class_rows.append({
        "label": class_id,
        "class": class_name,
        "count": int(mask.sum()),
        "accuracy": float(
            (val_predictions[mask] == class_id).mean()
        ),
    })

per_class = pd.DataFrame(class_rows)
per_class.to_csv(OUT / "validation_per_class.csv", index=False)

print(
    f"\nSelected epoch: {checkpoint['epoch']} | "
    f"Verified validation: {verified_accuracy:.2%}",
    flush=True,
)
print(per_class.to_string(index=False), flush=True)


# ============================================================
# 11. Evaluation predictions and submission
# ============================================================

# Evaluation images are used only for final inference.
del all_images, all_labels
del train_indices_gpu, val_indices_gpu
torch.cuda.empty_cache()

print("\nLoading evaluation images for inference...", flush=True)
test_images = load_images(test_df, "test_images")

test_images_gpu = torch.from_numpy(
    test_images.transpose(0, 3, 1, 2).copy()
).to(device)

del test_images

test_probabilities = predict(model, test_images_gpu)
test_predictions = test_probabilities.argmax(axis=1).astype(np.int64)

assert test_probabilities.shape == (len(test_df), 10)
assert np.isfinite(test_probabilities).all()
assert np.allclose(test_probabilities.sum(axis=1), 1, atol=1e-5)

np.save(OUT / "test_probs.npy", test_probabilities)
np.save(OUT / "test_ids.npy", test_df["id"].to_numpy())

# Match predictions to IDs, then use sample_submission order.
prediction_lookup = pd.Series(
    test_predictions,
    index=test_df["id"],
)

submission = sample_df[["id"]].copy()
submission["label"] = submission["id"].map(prediction_lookup)

assert submission["label"].notna().all()
submission["label"] = submission["label"].astype(np.int64)

assert submission.columns.tolist() == ["id", "label"]
assert submission["id"].tolist() == sample_df["id"].tolist()
assert submission["id"].is_unique
assert submission["label"].between(0, 9).all()
assert len(submission) == len(test_df)

submission_path = OUT / "submission.csv"
submission.to_csv(submission_path, index=False)

results = {
    "experiment": EXPERIMENT,
    "parameters": parameter_count,
    "selected_epoch": int(checkpoint["epoch"]),
    "validation_accuracy": verified_accuracy,
    "training_images": len(training_idx),
    "validation_images": len(validation_idx),
    "label_smoothing": LABEL_SMOOTHING,
    "color_jitter": 0.0,
    "weight_source": "ema",
    "horizontal_flip_averaging": True,
    "submission_rows": len(submission),
    "submission_sha256": hashlib.sha256(
        submission_path.read_bytes()
    ).hexdigest(),
}

(OUT / "results.json").write_text(json.dumps(results, indent=2))

print("\nCOMPLETE", flush=True)
print("Best checkpoint:", OUT / "best.pt", flush=True)
print("Recovery checkpoint:", OUT / "last.pt", flush=True)
print("Validation predictions:", OUT / "validation_predictions.csv")
print("Submission:", submission_path, flush=True)
print(submission.head(), flush=True)

display(FileLink(str(submission_path)))

GPU: Tesla T4
PyTorch: 2.11.0+cu128
Output: /kaggle/working/four_stage_cutout_ema_ls005_20261101_201953_270798
One GPU is used.
Label smoothing: 0.05
Data: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Training images: 50000
Evaluation images: 2000
Loading permitted training images...
Fitting images: 45000
Validation images: 5000
Mean: [0.49113735 0.48211119 0.44644899]
Std: [0.24686914 0.24343013 0.26160339]
Trainable parameters: 4,815,940
Parameter budget check: PASSED
Training from fresh random initialization.
Epoch 001/200 | LR 0.004000 | Loss 1.6205 | Val 63.80% | Best 63.80% | 26s | Remaining ~1.46h
Epoch 002/200 | LR 0.004263 | Loss 1.2818 | Val 72.98% | Best 72.98% | 15s | Remaining ~1.14h
Epoch 003/200 | LR 0.005049 | Loss 1.1529 | Val 77.82% | Best 77.82% | 15s | Remaining ~1.03h
Epoch 004/200 | LR 0.006350 | Loss 1.0589 | Val 81.22% | Best 81.22% | 15s | Remaining ~0.98h
Epoch 005/200 | LR 0.008151 | Loss 1.0070 | Val 83.12% | Best 83.12% | 15s | Rema

/kaggle/working/four_stage_cutout_ema_ls005_20261101_201953_270798/submission.csv